# Гибридная рекомендательная система книг (Goodbooks-10k): итоговый проект

Итоговый проект по рекомендательным системам. Продолжение домашнего задания №1 (`goodbooks_recommender.ipynb`): те же данные, тот же протокол разбиения и метрик, те же модели. Поэтому все цифры можно сравнивать с ДЗ напрямую.

**Что сделано**

| Часть | Что реализовано | Где |
|---|---|---|
| Этап 1 | Расширенные признаки пользователей, книг и взаимодействий, тематические категории | раздел 1 |
| Этап 2 | Гибрид: взвешенное усреднение, веса по типам пользователей, пул кандидатов, баланс релевантности и разнообразия, популярное, фильтрация прочитанного | раздел 2 |
| Этап 3 | Сравнение гибрида с одиночными моделями, анализ по сегментам, оптимизация весов, метрики, выводы | раздел 3 |
| Продвинутая часть | Нейросетевая Two-Tower модель, интеграция в гибрид, сквозной пайплайн с перезапуском на новых данных | разделы 4 и 5 |

**Организация кода.** Весь код вынесен в пакет `recsys/` (данные, признаки, метрики, модели, нейросеть, гибрид, пайплайн), а ноутбук - это отчёт: он вызывает этапы пайплайна по очереди и объясняет результаты. Так один и тот же код используется и здесь, и в `python run_pipeline.py`, и в тестах.

**Протокол оценки** (как в ДЗ). Временных меток нет, поэтому разбиение случайное внутри каждого пользователя: train 70%, val 10%, test 20%. Релевантной считается книга с оценкой >= 4. Кандидаты - все книги, которых нет в обучающей части пользователя. Гиперпараметры и веса гибрида подбираются на val (модели обучены на train), финальные модели обучаются на train+val и оцениваются на test один раз.

In [ ]:
import os, sys, time, warnings, dataclasses
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA

from recsys.data import LIKE_THRESHOLD, to_csr
from recsys.features import build_features, interaction_features, USER_SEGMENTS
from recsys.metrics import FastEvaluator, evaluate_ranking, paired_diff, results_table, ranking_metrics
from recsys.neural import TwoTowerConfig, TwoTowerRecommender
from recsys.hybrid import HybridRecommender, HybridSystem, BalanceConfig, item_confidence
from recsys.pipeline import (Pipeline, PipelineConfig, build_components, synthetic_dataset,
                             MODEL_TITLES, SEGMENT_TITLES)

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.labelsize": 11})
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 220)

SEED = 42
FAST = bool(os.environ.get("RECSYS_FAST"))          # быстрый режим на синтетических данных (для отладки ноутбука)
DATA_DIR = os.environ.get("RECSYS_DATA", "data")
OUT_DIR = os.environ.get("RECSYS_OUT", "artifacts/notebook_run")

cfg = PipelineConfig(data_dir=DATA_DIR, out_dir=OUT_DIR, seed=SEED)
if FAST:   # маленькие модели и выборки, чтобы прогнать ноутбук за минуту
    cfg.params = {"popularity": {"strategy": "mean_rating", "min_ratings": 3}, "item_cf": {"k": 5, "shrinkage": 0.0},
                  "svd": {"n_factors": 5, "n_epochs": 5, "reg_all": 0.1, "random_state": 0}, "pure_svd": {"n_factors": 20}}
    cfg.neural = TwoTowerConfig(epochs=3, batch_size=256, hidden=32, emb_dim=16, out_dim=16, hist_len=20)
    cfg.n_topics, cfg.emb_dim, cfg.tune_users_per_segment, cfg.search_random, cfg.search_sweeps = 5, 16, 60, 3, 1
pipe = Pipeline(cfg)
print("Режим:", "БЫСТРЫЙ (синтетика)" if FAST else "полный", "| данные:", DATA_DIR)

# 0. Данные и разбиение

Загрузка и очистка те же, что в ДЗ: удаляются дубли пар (пользователь, книга), ключи `books.id` и `books.book_id` переименованы, служебные «полочные» теги (`to-read`, `favorites`...) не используются. Проверка схемы входных файлов добавлена в `load_raw`, чтобы пайплайн падал сразу и понятно, если новые данные в другом формате.

In [ ]:
pipe.load()
ds = pipe.ds
n_r = len(ds.ratings)
display(pd.Series({"пользователей": ds.n_users, "книг": ds.n_items, "оценок": n_r,
                   "разреженность": 1 - n_r / (ds.n_users * ds.n_items),
                   "доля оценок >= 4": (ds.ratings["rating"] >= 4).mean()}, name="значение").to_frame())

fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
ds.ratings["rating"].value_counts(normalize=True).sort_index().plot.bar(ax=ax[0], color="C0")
ax[0].set(title="Распределение оценок", xlabel="оценка", ylabel="доля")
cnt = ds.ratings.groupby("u").size()
ax[1].hist(np.log10(cnt), bins=30, color="C1", edgecolor="white")
ax[1].set(title="Активность пользователей", xlabel="log10(число оценок)", ylabel="пользователей")
ax[2].hist(ds.books["average_rating"], bins=30, color="C2", edgecolor="white")
ax[2].set(title="Средняя оценка книг (Goodreads)", xlabel="средняя оценка", ylabel="книг")
plt.tight_layout(); plt.show()

Выводы EDA из ДЗ, которые определяют дизайн проекта (подробности и графики - в первом ноутбуке):

- **Разреженность** около 99.8%, **тяжёлый хвост активности**: у трети пользователей меньше 5 оценок, а активные (50+) дают 43% всех оценок. Поэтому пользователей нужно делить на типы и вести себя с ними по-разному.
- **Смещение к высоким оценкам** (две трети оценок >= 4): средняя оценка книги плохо различает книги, порог релевантности равен 4.
- **Популярность внутри выборки неинформативна** (не больше 100 оценок на книгу), поэтому Popularity в ДЗ не лучше случайной выдачи.
- **Локальный сигнал**: у 39% понравившихся в test книг тот же автор, что у понравившейся пользователю книги из train, у 23% - та же серия. Это идея для признаков взаимодействия.

In [ ]:
pipe.prepare()
sp = pipe.splits
split_tbl = pd.DataFrame({
    "оценок": [len(d) for d in (sp.train_df, sp.val_df, sp.trainval_df, sp.test_df)],
    "релевантных (>=4)": [(d["rating"] >= 4).sum() for d in (sp.train_df, sp.val_df, sp.trainval_df, sp.test_df)],
    "пользователей": [d["u"].nunique() for d in (sp.train_df, sp.val_df, sp.trainval_df, sp.test_df)],
}, index=["train", "val", "trainval", "test"])
split_tbl["доля"] = split_tbl["оценок"] / n_r
display(split_tbl)

# Этап 1. Расширенные признаки

Признаки двух типов (модуль `recsys/features.py`):

- **Статические признаки книг** не зависят от оценок: тематические категории, текстовые эмбеддинги, автор, серия, метаданные Goodreads. Считаются один раз.
- **Признаки, зависящие от оценок** (пользователи, популярность и качество книг) считаются **только по обучающей части**: для подбора на val - по train, для финальной оценки - по train+val. Утечки из test нет.

## 1.1 Тематические категории книг

Ручной словарь жанров не нужен: матрица «книга x содержательный тег» (1500 самых частых несервисных тегов, вес `log(1+count)`) раскладывается NMF на 20 тем. Каждая тема получает имя по трём самым характерным тегам, а книга - вектор принадлежности темам (сумма равна 1). Тот же подход работает на любых новых данных с тегами.

In [ ]:
content = pipe.content
main_genre = content.genre.argmax(axis=1)
gtab = pd.DataFrame({"тема (топ-теги)": content.genre_names,
                     "книг, где тема главная": np.bincount(main_genre, minlength=content.n_genres),
                     "средний вес темы": content.genre.mean(axis=0)})
display(gtab.sort_values("книг, где тема главная", ascending=False).reset_index(drop=True))
print("Средняя доля главной темы в профиле книги: %.2f (1.0 - книга целиком в одной теме)" % content.genre.max(axis=1).mean())

## 1.2 Признаки пользователей

По обучающей выборке для каждого пользователя считаются: число оценок и перцентиль активности, средняя оценка, смещение относительно общей средней, разброс оценок, доли лайков и дизлайков, «мейнстримность» (средняя популярность прочитанных книг по Goodreads), широта вкуса (энтропия тем понравившихся книг), доля книг из серий и повторяемость авторов. По числу оценок пользователь относится к одному из трёх **типов**: новый (< 5 оценок), обычный (5-29), активный (30+). Эти типы используются на этапе 2.

In [ ]:
uf = pipe.store_train.user_features.copy()
uf["тип"] = uf["segment"].map(SEGMENT_TITLES)
show = ["n_ratings", "mean_rating", "rating_std", "like_share", "mainstream", "genre_entropy", "series_share", "author_repeat"]
tbl = uf.groupby("тип")[show].mean().loc[[SEGMENT_TITLES[s] for s in range(3)]]
tbl.insert(0, "доля пользователей", uf["тип"].value_counts(normalize=True).reindex(tbl.index))
display(tbl)

fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
order = [SEGMENT_TITLES[s] for s in range(3)]
for a, col, title in zip(ax, ["mean_rating", "genre_entropy", "series_share", "mainstream"],
                         ["Средняя оценка пользователя", "Широта вкуса (энтропия тем)", "Доля книг из серий", "Мейнстримность"]):
    sns.boxplot(data=uf, x="тип", y=col, order=order, ax=a, showfliers=False, palette="Set2")
    a.set(title=title, xlabel="", ylabel=""); a.tick_params(axis="x", labelrotation=15)
plt.tight_layout(); plt.show()

Новые пользователи по профилю резко отличаются: у них более «крайние» оценки (средняя ближе к 5 и 1), вкус по нескольким книгам выглядит узким, поэтому энтропия тем у них ниже. Активные читатели шире по жанрам. Это согласуется с идеей использовать для разных типов разные комбинации моделей.

## 1.3 Признаки книг

По обучающей выборке: число оценок и перцентиль популярности, байесовская средняя оценка (со сжатием к общей средней, чтобы книга с одной пятёркой не обгоняла остальных), разброс и **энтропия распределения оценок** («разнообразие оценок»: низкая у книг, которые все оценивают одинаково, высокая у спорных), доля лайков. Из метаданных Goodreads: логарифм числа оценок, средняя оценка, энтропия оценок по всему Goodreads, год издания. Плюс тематический профиль из 1.1, автор и серия.

In [ ]:
itf = pipe.store_train.item_features
cols = ["log_n_ratings", "bayes_mean", "rating_std", "rating_entropy", "like_ratio", "log_gr_count", "gr_avg_rating", "gr_rating_entropy", "year"]
display(itf[cols].describe().T[["mean", "std", "min", "50%", "max"]])

fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
ax[0].scatter(itf["log_gr_count"], itf["bayes_mean"], s=3, alpha=0.3)
ax[0].set(xlabel="log(число оценок на Goodreads)", ylabel="байесовская средняя (train)", title="Популярность и качество книг")
ax[1].scatter(itf["rating_entropy"], itf["rating_std"], s=3, alpha=0.3, color="C1")
ax[1].set(xlabel="энтропия оценок (train)", ylabel="стд. оценок (train)", title="Разнообразие оценок книги")
corr = itf[cols].corr()
sns.heatmap(corr, ax=ax[2], cmap="vlag", center=0, cbar=False, xticklabels=range(len(cols)), yticklabels=range(len(cols)))
ax[2].set(title="Корреляции признаков книг (номера = порядок в таблице)")
plt.tight_layout(); plt.show()

## 1.4 Признаки взаимодействий «пользователь - книга»

Для пары (пользователь u, кандидат i) считаются признаки близости к истории пользователя (его лайки в train):

- `same_author_likes`, `same_series_likes`: сколько понравившихся книг написал тот же автор / входит в ту же серию;
- `max_history_sim`: косинус текстовых эмбеддингов до самой близкой понравившейся книги;
- `genre_affinity`: косинус тематического профиля пользователя и книги;
- `pop_gap`: расстояние между популярностью книги и «мейнстримностью» пользователя;
- `item_bias_vs_user`: байесовская средняя книги минус средняя оценка пользователя.

Проверим, что они несут сигнал: возьмём пары (пользователь, понравившаяся ему в val книга) как положительные и такое же число случайных непрочитанных книг как отрицательные, и посчитаем ROC AUC каждого признака отдельно (0.5 - признак бесполезен).

In [ ]:
rng = np.random.default_rng(SEED)
val_pos = sp.val_df[sp.val_df["rating"] >= LIKE_THRESHOLD]
val_pos = val_pos.sample(min(20000, len(val_pos)), random_state=SEED)
neg_i = rng.integers(0, ds.n_items, len(val_pos))
seen_pair = np.asarray(sp.R_train[val_pos["u"].to_numpy(), neg_i]).ravel() > 0
users = np.concatenate([val_pos["u"].to_numpy(), val_pos["u"].to_numpy()[~seen_pair]])
items = np.concatenate([val_pos["i"].to_numpy(), neg_i[~seen_pair]])
labels = np.concatenate([np.ones(len(val_pos)), np.zeros((~seen_pair).sum())])
has_hist = np.asarray(pipe.store_train.likes[users].sum(axis=1)).ravel() > 0         # без истории признаки взаимодействия нулевые
inter = interaction_features(pipe.store_train, users[has_hist], items[has_hist])
lab = labels[has_hist]
auc_tbl = pd.DataFrame({
    "среднее у понравившихся": inter[lab == 1].mean(), "среднее у случайных": inter[lab == 0].mean(),
    "ROC AUC": {c: roc_auc_score(lab, inter[c]) for c in inter.columns}})
display(auc_tbl.sort_values("ROC AUC", ascending=False))

Самые сильные признаки взаимодействия - близость к истории по тексту и совпадение автора или серии: именно они ловят «локальный» сигнал, который мы видели в EDA. Признаки «популярность» и «смещение книги относительно пользователя» заметно слабее (AUC около 0.5-0.6): в этой выборке они почти не различают книги, что согласуется с поведением Popularity в ДЗ.

**Как признаки используются в системе** (ниже по тексту это проверяется измерениями, а не утверждается):

1. автор/серия - самостоятельная модель-компонента гибрида (`AuthorSeriesAffinity`), а текстовая близость и жанры - вход нейросети и MMR-переранжирования;
2. признаки пользователей задают **тип пользователя** (новый / обычный / активный) и от него зависят веса гибрида;
3. признаки пользователей и книг - вход башен Two-Tower, раздел 4 содержит абляцию «с признаками и без»;
4. приор качества книги (`ItemQualityPrior`) заменяет роль смещения `b_i` из SVD.

# Этап 2. Гибридная система

## 2.1 Компоненты

| Ключ | Модель | Что даёт |
|---|---|---|
| `pop` | Popularity (средняя оценка с порогом) | неперсональный запасной вариант |
| `content` | Content-based (TF-IDF) | работает на книгах без оценок |
| `cf` | Item-based CF (implicit) | лучшая одиночная модель из ДЗ |
| `svd` | SVD (surprise) | оценка качества книги через смещения |
| `pure` | PureSVD (implicit) | глобальные факторы, сильна у активных пользователей |
| `meta` | автор/серия (признаки взаимодействия) | «следующий том серии» |
| `prior` | приор качества книги | неперсональная поправка на качество |
| `neural` | Two-Tower | нейросетевая модель, раздел 4 |

Гиперпараметры моделей из ДЗ взяты как есть (подобраны там на val): `Item-CF K=10`, `PureSVD f=800`, `SVD f=20`, порог Popularity 40. В `Pipeline.tune_hyperparams` есть автоматический подбор по сетке для запуска на новых данных.

In [ ]:
print(pd.Series({k: v for k, v in pipe.cfg.params.items()}).to_string())
pipe.fit_train_models()      # все модели на train: нужны для подбора весов и баланса на val

## 2.2 Стратегии объединения

**Взвешенное усреднение.** Скоры моделей имеют разные шкалы, поэтому для каждого пользователя они стандартизуются (z-score по всем книгам), затем

$$\text{score}(u,i)=c_i\sum_m w_m z_m(u,i)+(1-c_i)\,z_{\text{content}}(u,i),\qquad c_i=\frac{n_i}{n_i+c}.$$

$c_i$ - доверие к коллаборативным сигналам книги: у книги без оценок $c_i=0$ и её ранжирует только контент. Это схема гибрида из ДЗ.

**Разделение пользователей по типам.** Веса $w_m$ подбираются отдельно для новых (< 5 оценок), обычных (5-29) и активных (30+) пользователей: у новичков коллаборативные модели знают мало, у активных пользователей CF и PureSVD работают лучше всего.

**Подбор весов.** Сетка с шагом 0.1 по 8 моделям - это тысячи комбинаций, поэтому используется поиск: старт из чистых моделей, равных весов и случайных точек симплекса, затем покоординатный локальный поиск. Целевая функция - nDCG@10 на val; для общих весов - сумма по типам пользователей в естественных пропорциях. Чтобы не подгонять веса под шум, улучшение принимается только если оно больше порога `min_gain = 0.0005`, а веса меньше 3% обнуляются. Для честности веса подбираются на моделях, обученных **на train**, и применяются к моделям, переобученным на train+val.

Рассмотрим четыре варианта гибрида, каждый следующий добавляет одну идею:

1. **Hybrid-1** - модели из ДЗ (`pop, svd, pure, cf, content`), общие веса;
2. **Hybrid-2** - плюс признаки (`meta`, `prior`), общие веса;
3. **Hybrid-3** - те же компоненты, веса по типам пользователей;
4. **Hybrid-4** - плюс нейросетевая `neural`, веса по типам (и вариант 4a с общими весами).

In [ ]:
pipe.tune_hybrid()
seg_n = {SEGMENT_TITLES[s]: f"{pipe.segment_share[s]:.0%}" for s in pipe.segment_share}
print("Доля типов среди пользователей val (веса общей цели):", seg_n)
display(Markdown("**Качество одиночных моделей на val (nDCG@10, пользователи для подбора весов)**"))
display(pipe.val_single.rename(index=MODEL_TITLES))

In [ ]:
rows = {}
labels_v = {"hw": "Hybrid-1 (общие)", "feat": "Hybrid-2 (общие)", "nn": "Hybrid-4a (общие)"}
for v, lab in labels_v.items():
    if v in pipe.weights:
        rows[lab] = pipe.weights[v][None]
for v, lab in {"feat_seg": "Hybrid-3", "nn_seg": "Hybrid-4"}.items():
    if v in pipe.weights:
        for s in range(3):
            rows[f"{lab}: {SEGMENT_TITLES[s]}"] = pipe.weights[v][s]
W = pd.DataFrame(rows).T.fillna(0.0)
W = W[[c for c in ["pop", "svd", "pure", "cf", "content", "meta", "prior", "neural"] if c in W.columns]]
fig, ax = plt.subplots(figsize=(9, 0.45 * len(W) + 1.5))
sns.heatmap(W, annot=True, fmt=".2f", cmap="Blues", cbar=False, ax=ax, vmin=0, vmax=1)
ax.set(title="Подобранные веса гибрида (сумма по строке равна 1)")
plt.tight_layout(); plt.show()

Читать веса нужно осторожно. Скоры стандартизованы по пользователю, но у разреженных моделей (CF, PureSVD, автор/серия) распределение z-скоров с тяжёлым хвостом, а у плотных (нейросеть, контент) оно ближе к нормальному, поэтому вес - это смесь «важности» и «масштаба» компоненты. Надёжный вывод даёт качество на отложенной выборке (раздел 3), а не абсолютные значения весов. Качественно видно, например, что у активных пользователей растёт вес PureSVD (глобальные факторы начинают работать при длинной истории), а у новых - вес контентных и неперсональных компонент.

In [ ]:
# чувствительность гибрида к весу нейросети и сходимость поиска
key = [k for k in pipe.searches if len(k[0]) == 3 and "neural" in k[1]]
if key:
    hist = pipe.searches[key[0]].history
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
    sc = ax[0].scatter(hist["neural"], hist["nDCG@10"], c=hist["cf"], cmap="viridis", s=14)
    ax[0].set(xlabel="вес Two-Tower", ylabel="nDCG@10 (val)", title="Все проверенные комбинации весов (цвет - вес Item-CF)")
    plt.colorbar(sc, ax=ax[0])
    ax[1].plot(hist["nDCG@10"].cummax().values)
    ax[1].set(xlabel="число проверенных комбинаций", ylabel="лучший nDCG@10 (val)", title="Сходимость подбора весов")
    plt.tight_layout(); plt.show()

## 2.3 Генерация кандидатов и балансировка выдачи

Итоговая система (`HybridSystem`) работает в три шага:

1. **Единый пул кандидатов.** Из каждой модели берутся top-N книг (CF - 50, PureSVD - 30, контент - 30, автор/серия - 20, Two-Tower - 30) и ещё 20 популярных. Объединение - пул. **Прочитанные книги исключаются** до формирования пула.
2. **Гибридный скор** (веса по типу пользователя) считается для книг пула.
3. **Баланс релевантности и разнообразия:** MMR-переранжирование top-50 пула
   $$\arg\max_{i\notin S}\ \lambda\,\widehat{\text{rel}}(i)-(1-\lambda)\max_{j\in S}\cos(e_i,e_j),$$
   где $e_i$ - текстовые эмбеддинги книг, а $\widehat{\text{rel}}$ - гибридный скор, приведённый к [0, 1] внутри списка. Дополнительно часть последних позиций выдачи можно отдать неперсональному «популярному» (долю подбираем отдельно для каждого типа пользователей). Пользователь без истории получает только популярное.

Параметры выбираются по val по явному правилу: **берём наименьшее $\lambda$ (самое сильное разнообразие), при котором nDCG@10 падает не больше чем на 2% от значения без переранжирования.** Доля популярного выбирается по типам пользователей по nDCG@10.

In [ ]:
pipe.tune_balance()
bt = pipe.balance_table
display(bt.rename(columns={"lambda": "λ (1 = без разнообразия)"}))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(bt["lambda"], bt["nDCG@10"], "o-", label="nDCG@10")
ax[0].plot(bt["lambda"], bt["R@10"], "s-", label="Recall@10")
ax[0].axvline(pipe.mmr_lambda, color="r", ls="--", label=f"выбрано λ = {pipe.mmr_lambda}")
ax[0].invert_xaxis(); ax[0].legend(); ax[0].set(xlabel="λ", title="Релевантность при усилении разнообразия (val)")
ax[1].plot(bt["ild@10"], bt["nDCG@10"], "o-")
for _, r in bt.iterrows():
    ax[1].annotate(f"λ={r['lambda']}", (r["ild@10"], r["nDCG@10"]), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax[1].set(xlabel="ILD@10 (разнообразие внутри выдачи)", ylabel="nDCG@10", title="Компромисс релевантность - разнообразие")
plt.tight_layout(); plt.show()
display(Markdown("**Доля слотов под популярное** (по типам пользователей, val)"))
display(pipe.pop_share_table.pivot(index="сегмент", columns="доля популярного", values="nDCG@10"))
print("Выбрано:", {SEGMENT_TITLES[s]: v for s, v in pipe.pop_share.items()}, "| λ =", pipe.mmr_lambda)

# Этап 3. Оценка и оптимизация

Финальное обучение на train+val и оценка на test. Метрики те же, что в ДЗ:

- Precision@K = (число релевантных в top-K) / K;
- Recall@K = (число релевантных в top-K) / (число релевантных у пользователя в test);
- nDCG@K = DCG@K / IDCG@K, $\mathrm{DCG@K}=\sum_{r=1}^{K}\frac{\mathrm{rel}_r}{\log_2(r+1)}$.

Корректность реализации проверена в `tests/test_pipeline.py` на примерах, посчитанных вручную. Дополнительно считаются метрики выдачи в целом: coverage@K (доля каталога в выдачах), novelty@K (средняя «редкость» книг, $-\log_2 p_i$), ILD@K (разнообразие внутри выдачи, 1 - средний косинус эмбеддингов).

In [ ]:
pipe.fit_final()
pipe.evaluate()
results = pipe.results
print("Пользователей в оценке:", int(next(iter(results.values())).summary["n_users"]))

## 3.1 Сравнение гибрида с одиночными моделями

In [ ]:
main_cols = ["P@10", "R@10", "nDCG@10", "coverage@10", "novelty@10", "ild@10"]
summary_tbl = results_table(results, main_cols)
display(Markdown("### Сводная таблица (test, K = 10)"))
display(summary_tbl.style.format("{:.4f}").background_gradient(axis=0, subset=["P@10", "R@10", "nDCG@10"], cmap="Greens"))

KS = pipe.cfg.ks
full_tbl = pd.concat({m: pd.DataFrame({name: r.summary for name, r in results.items()}).T[[f"{m}@{k}" for k in KS]]
                      .set_axis([f"K={k}" for k in KS], axis=1) for m in ("P", "R", "nDCG")}, axis=1)
display(Markdown("### Все метрики при K = 5, 10, 20"))
display(full_tbl)

In [ ]:
plot_models = [m for m in results if not m.startswith("Random")]
fig, ax = plt.subplots(1, 3, figsize=(19, 6.2))
for a, metric, title in zip(ax, ("P", "R", "nDCG"), ("Precision@10", "Recall@10", "nDCG@10")):
    vals = [results[m].summary[f"{metric}@10"] for m in plot_models]
    colors = ["C3" if m.startswith("Hybrid") else "C0" for m in plot_models]
    bars = a.barh(plot_models, vals, color=colors)
    a.axvline(results["Random (reference)"].summary[f"{metric}@10"], color="gray", ls="--")
    for b, v in zip(bars, vals):
        a.text(v, b.get_y() + b.get_height() / 2, f" {v:.3f}", va="center", fontsize=8)
    a.set(title=title); a.invert_yaxis()
    if metric != "P": a.set_yticklabels([])
plt.suptitle("Сравнение моделей на test, K = 10 (красные - гибриды)", y=1.0)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
show_models = ["Content-based (TF-IDF)", "Item-based CF (implicit)", "PureSVD (implicit)", "Two-Tower (neural)",
               "Hybrid-1: модели ДЗ, общие веса", "Hybrid system (пул + баланс)"]
for a, metric in zip(ax, ("P", "R", "nDCG")):
    for m in show_models:
        a.plot(KS, [results[m].summary[f"{metric}@{k}"] for m in [m] for k in KS], "o-", label=m)
    a.set(xlabel="K", ylabel=f"{metric}@K", title=f"{metric}@K", xticks=KS)
ax[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

### Вклад каждой идеи (поэтапное сравнение)

Парное сравнение по пользователям (разность метрики и 95% доверительный интервал): значимой считается разность, у которой интервал не содержит нуля. Сравниваем каждый шаг с предыдущим и с лучшей одиночной моделью из ДЗ.

In [ ]:
chain = ["Item-based CF (implicit)", "Hybrid-1: модели ДЗ, общие веса", "Hybrid-2: + признаки (автор/серия, приор)",
         "Hybrid-3: + веса по сегментам", "Hybrid-4: + Two-Tower, веса по сегментам", "Hybrid system (пул + баланс)"]
chain = [c for c in chain if c in results]
rows = []
for prev, cur in zip(chain[:-1], chain[1:]):
    for metric in ("nDCG@10", "R@10"):
        d = paired_diff(results[cur], results[prev], metric)
        rows.append({"шаг": f"{cur}  vs  {prev}", "метрика": metric, **d.to_dict()})
for metric in ("nDCG@10", "R@10"):
    d = paired_diff(results[chain[-1]], results[chain[0]], metric)
    rows.append({"шаг": f"ИТОГО: {chain[-1]}  vs  {chain[0]}", "метрика": metric, **d.to_dict()})
display(pd.DataFrame(rows).set_index(["шаг", "метрика"]))

fig, ax = plt.subplots(figsize=(10, 3.8))
vals = [results[c].summary["nDCG@10"] for c in chain]
ax.plot(range(len(chain)), vals, "o-")
for i, v in enumerate(vals):
    ax.annotate(f"{v:.4f}", (i, v), xytext=(0, 7), textcoords="offset points", ha="center", fontsize=9)
ax.set_xticks(range(len(chain))); ax.set_xticklabels([c.split(":")[0] if c.startswith("Hybrid-") else c.replace(" (implicit)", "") for c in chain], rotation=15)
ax.set(ylabel="nDCG@10 (test)", title="От лучшей одиночной модели к итоговой системе")
plt.tight_layout(); plt.show()

## 3.2 Анализ по сегментам пользователей

In [ ]:
seg_tbl = pipe.metrics_by_segment("nDCG@10")
cnt_by_seg = pd.Series(pipe.store_tv.user_segment[results["Item-based CF (implicit)"].per_user.index]).map(SEGMENT_TITLES).value_counts()
display(Markdown("**nDCG@10 по типам пользователей (test)**; пользователей в оценке: " + ", ".join(f"{k} - {v}" for k, v in cnt_by_seg.items())))
display(seg_tbl.loc[[m for m in seg_tbl.index if not m.startswith("Random")]])

show = ["Content-based (TF-IDF)", "Item-based CF (implicit)", "PureSVD (implicit)", "Two-Tower (neural)",
        "Hybrid-1: модели ДЗ, общие веса", "Hybrid system (пул + баланс)"]
ax = seg_tbl.loc[show].T.plot.bar(figsize=(11, 4.2), rot=0, width=0.8)
ax.set(ylabel="nDCG@10", title="Качество по типам пользователей"); ax.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

act_bins = [0, 3, 6, 12, 25, 50, 10**6]; act_labels = ["1-3", "4-6", "7-12", "13-25", "26-50", "51+"]
fig, ax = plt.subplots(figsize=(9, 4.2))
for m in show:
    pu = results[m].per_user
    ax.plot(act_labels, pu.groupby(pd.cut(pu["n_train"], act_bins, labels=act_labels))["nDCG@10"].mean().values, "o-", label=m)
ax.set(xlabel="число оценок пользователя в train+val", ylabel="nDCG@10", title="Качество и активность пользователя")
ax.legend(fontsize=7); plt.tight_layout(); plt.show()

## 3.3 Релевантность и разнообразие: что меняет балансировка

In [ ]:
div_cols = ["nDCG@10", "R@10", "ild@10", "coverage@10", "novelty@10"]
cmp_ = ["Item-based CF (implicit)", "Content-based (TF-IDF)", "Two-Tower (neural)",
        "Hybrid-4: + Two-Tower, веса по сегментам", "Hybrid system (пул + баланс)"]
cmp_ = [c for c in cmp_ if c in results]
display(results_table(results, div_cols).loc[cmp_])

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for m in results:
    if m.startswith(("Random", "Popularity", "SVD", "Item quality")):
        continue
    s = results[m].summary
    ax.scatter(s["ild@10"], s["nDCG@10"], s=70, color="C3" if m.startswith("Hybrid") else "C0")
    ax.annotate(m.split(":")[0].replace(" (implicit)", "").replace(" (neural)", ""), (s["ild@10"], s["nDCG@10"]), fontsize=7,
                xytext=(4, 3), textcoords="offset points")
ax.set(xlabel="ILD@10 (разнообразие выдачи)", ylabel="nDCG@10", title="Релевантность и разнообразие на test")
plt.tight_layout(); plt.show()

In [ ]:
# статистика пула кандидатов: сколько книг в пуле и какую долю релевантных книг test он «захватывает»
rng = np.random.default_rng(SEED)
rel_u = np.unique(sp.test_df.loc[sp.test_df["rating"] >= LIKE_THRESHOLD, "u"])
us = np.sort(rng.choice(rel_u, min(2000, len(rel_u)), replace=False))
seen = sp.R_trainval[us]
pool = pipe.system.pool_mask(us, seen)
rel = to_csr(sp.test_df[sp.test_df["rating"] >= LIKE_THRESHOLD], ds.shape, value=None, dtype=np.int8)[us].toarray() > 0
top50 = pipe.system.hybrid.score(us); top50[seen.toarray() > 0] = -np.inf
from recsys.metrics import top_k
t10, t50 = top_k(top50, 10), top_k(top50, 50)
r_ = lambda idx: np.take_along_axis(rel, idx, 1).sum() / rel.sum()
display(pd.Series({"средний размер пула (книг)": pool.sum(1).mean(),
                   "доля релевантных книг test, попавших в пул": (pool & rel).sum() / rel.sum(),
                   "то же для top-10 гибрида": r_(t10), "то же для top-50 гибрида": r_(t50),
                   "доля пула в каталоге": pool.sum(1).mean() / ds.n_items}, name="значение").to_frame())

## 3.4 Холодный старт книг

Как в ДЗ: случайные 10% каталога полностью убираются из обучения, все модели (включая нейросеть) переобучаются, и мы ранжируем только эти «новые» книги. Релевантны те из них, которые пользователь действительно оценил на 4 и выше. Для гибрида сравниваем два варианта работы с книгами без оценок: по одному контенту (как в ДЗ) и по смеси контента и нейросети (у Two-Tower есть контентная башня, обученная с dropout ID-эмбеддинга).

In [ ]:
rng = np.random.default_rng(7)
n_cold = max(10, ds.n_items // 10)
cold_items = np.sort(rng.choice(ds.n_items, n_cold, replace=False))
is_cold = np.zeros(ds.n_items, dtype=bool); is_cold[cold_items] = True
warm_df = sp.trainval_df[~is_cold[sp.trainval_df["i"].to_numpy()]]
R_warm = to_csr(warm_df, ds.shape)
store_warm = build_features(ds, warm_df, pipe.content)
cold_ratings = ds.ratings[is_cold[ds.ratings["i"].to_numpy()]]

ncfg = dataclasses.replace(pipe.cfg.neural, epochs=pipe.neural_best_epoch)
models_cold = build_components(pipe.cfg, ds, warm_df, R_warm, store_warm, neural_cfg=ncfg)
counts_warm = np.asarray(R_warm.getnnz(axis=0)).ravel()
wv = "nn_seg" if pipe.cfg.use_neural else "feat_seg"
hyb_a = HybridRecommender(models_cold, pipe.weights[wv], counts_warm, store_warm.user_segment, c=pipe.cfg.conf_c)
cold_models = {MODEL_TITLES[k]: m for k, m in models_cold.items()}
cold_models["Hybrid-4 (холодные книги: контент)"] = hyb_a
if pipe.cfg.use_neural:
    cold_models["Hybrid-4 (холодные книги: контент + Two-Tower)"] = HybridRecommender(
        models_cold, pipe.weights[wv], counts_warm, store_warm.user_segment, c=pipe.cfg.conf_c,
        cold={"content": 0.5, "neural": 0.5})
results_cold = {n: evaluate_ranking(m, R_warm, cold_ratings, ks=KS, candidates=cold_items) for n, m in cold_models.items()}
cold_tbl = results_table(results_cold, ["P@10", "R@10", "nDCG@10", "coverage@10"])
display(Markdown(f"Пользователей в оценке: {int(next(iter(results_cold.values())).summary['n_users'])}; книг-кандидатов: {len(cold_items)}; "
                 f"число оценок новых книг в обучении: {int(counts_warm[cold_items].sum())}"))
display(cold_tbl.style.format("{:.4f}").background_gradient(axis=0, subset=["P@10", "R@10", "nDCG@10"], cmap="Greens"))
fig, ax = plt.subplots(figsize=(9, 4))
vals = cold_tbl["nDCG@10"]
bars = ax.barh(vals.index, vals.values, color=["C3" if i.startswith("Hybrid") else "C0" for i in vals.index])
ax.set(xlabel="nDCG@10", title="Рекомендации новых книг (без оценок в обучении)"); ax.invert_yaxis()
for b, v in zip(bars, vals.values):
    ax.text(v, b.get_y() + b.get_height() / 2, f" {v:.3f}", va="center", fontsize=8)
plt.tight_layout(); plt.show()

# 4. Продвинутая часть: нейросетевая Two-Tower модель

## 4.1 Архитектура

Выбран вариант **Two-Tower** (двухбашенная модель совместных эмбеддингов), модуль `recsys/neural.py`:

```
История лайков пользователя ──► усреднение ID-эмбеддингов ┐
                              ► усреднение текстовых эмбеддингов ├─► MLP ─► вектор пользователя (64, L2-норма) ─┐
Признаки пользователя (9 шт.) ───────────────────────────┘                                                      ├─► cos / τ
ID книги ─► ID-эмбеддинг (с dropout) ┐                                                                          │
Текстовый эмбеддинг книги (SVD от TF-IDF) ├─► MLP ─► вектор книги (64, L2-норма) ────────────────────────────────┘
Признаки книги (10 шт.) + тематический профиль ┘
```

- **Башня пользователя** не хранит вектор пользователя в таблице: он строится из истории лайков и признаков. Новому пользователю достаточно передать его лайки, переобучать модель не нужно.
- **Башня книги** использует ID-эмбеддинг, контент и признаки. С вероятностью 0.3 ID-эмбеддинг обнуляется при обучении, поэтому башня учится работать по контенту и признакам - это даёт качество на книгах без оценок.
- **Обучение.** Пары (пользователь, понравившаяся книга). Целевая книга **вырезается из истории** (иначе сеть «подсматривает» ответ), 30% оставшейся истории случайно скрывается (history dropout). Функция потерь - softmax по книгам из других пар батча (in-batch negatives) с **logQ-коррекцией** частот (чтобы популярные книги не доминировали в негативах), температура 0.07, Adam, косинусное расписание lr.
- **Инференс.** Векторы книг считаются один раз, скоры пользователя - одно матричное произведение, поэтому эмбеддинги можно положить в ANN-индекс.
- **Выбор числа эпох** - по nDCG@10 на val; финальная модель на train+val обучается с найденным числом эпох.

## 4.2 Обучение

In [ ]:
hist = pipe.neural_history
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(hist["epoch"], hist["loss"], "o-"); ax[0].set(xlabel="эпоха", ylabel="in-batch softmax loss", title="Функция потерь на train")
ax[1].plot(hist["epoch"], hist["val_nDCG@10"], "o-", color="C1")
ax[1].axvline(pipe.neural_best_epoch, color="r", ls="--", label=f"лучшая эпоха = {pipe.neural_best_epoch}")
ax[1].set(xlabel="эпоха", ylabel="nDCG@10 (val)", title="Качество на val"); ax[1].legend()
plt.tight_layout(); plt.show()
print("Параметров в модели:", sum(p.numel() for p in pipe.models["neural"].net.parameters()))

## 4.3 Абляция: что даёт контент и признаки

Обучаем варианты сети на train и сравниваем на тех же val-пользователях (12 эпох каждому, значение на последней эпохе, чтобы не выбирать эпоху отдельно для каждого варианта).

In [ ]:
rel_val = np.unique(sp.val_df.loc[sp.val_df["rating"] >= LIKE_THRESHOLD, "u"])
ev_abl = FastEvaluator(sp.R_train, sp.val_df, np.sort(np.random.default_rng(1).choice(rel_val, min(3000, len(rel_val)), replace=False)))
E = 3 if FAST else 12
base = dataclasses.replace(pipe.cfg.neural, epochs=E)
variants = {
    "только ID (история + ID книги)": dataclasses.replace(base, use_content=False, use_features=False),
    "ID + контент": dataclasses.replace(base, use_features=False),
    "контент + признаки (без ID)": dataclasses.replace(base, use_id=False),
    "ID + контент + признаки (полная)": base,
}
rows = {}
for name, vcfg in variants.items():
    t0 = time.time()
    m = TwoTowerRecommender(vcfg).fit(sp.R_train, pipe.store_train, evaluator=ev_abl)
    rows[name] = {"nDCG@10 (val, последняя эпоха)": m.history_[-1]["val_nDCG@10"],
                  "nDCG@10 (val, лучшая эпоха)": max(r["val_nDCG@10"] for r in m.history_), "время, с": time.time() - t0}
abl = pd.DataFrame(rows).T
display(abl)

Что видно из абляции: контент и признаки в башнях улучшают качество по сравнению с чистыми ID-эмбеддингами, а модель без ID вообще (только контент и признаки) сильно слабее по общему качеству, но она единственная полностью работоспособна на книгах без оценок. В полной модели ID-путь даёт качество на известных книгах, контентный путь - обобщение на новые.

## 4.4 Что выучила сеть

In [ ]:
nn = pipe.models["neural"]
iv = nn.item_vecs_.numpy()
n_pts = min(3000, ds.n_items)
sel = np.random.default_rng(0).choice(ds.n_items, n_pts, replace=False)
xy = PCA(2, random_state=0).fit_transform(iv[sel])
top_genres = pd.Series(main_genre[sel]).value_counts().index[:8]
fig, ax = plt.subplots(figsize=(8.5, 6))
other = ~np.isin(main_genre[sel], top_genres)
ax.scatter(xy[other, 0], xy[other, 1], s=4, color="lightgray", label="остальные темы")
for g, c in zip(top_genres, sns.color_palette("tab10", 8)):
    m = main_genre[sel] == g
    ax.scatter(xy[m, 0], xy[m, 1], s=7, color=c, label=content.genre_names[g][:28])
ax.legend(fontsize=7, markerscale=2); ax.set(title="Эмбеддинги книг из башни книги (PCA), цвет - главная тема", xticks=[], yticks=[])
plt.tight_layout(); plt.show()

def similar_by_neural(item: int, n: int = 5) -> pd.DataFrame:
    sims = iv @ iv[item]; sims[item] = -np.inf
    idx = np.argsort(-sims)[:n]
    return ds.books.loc[idx, ["title", "authors"]].assign(cos=sims[idx]).reset_index(drop=True)

for prefix in ["The Hunger Games", "Harry Potter and the Sorcerer's Stone", "Pride and Prejudice"]:
    hit = ds.books.index[ds.books["title"].str.startswith(prefix)]
    item = int(hit[0]) if len(hit) else int(ds.books["ratings_count"].idxmax())
    display(Markdown(f"**Ближайшие по Two-Tower к «{ds.books.loc[item, 'title']}»**"))
    display(similar_by_neural(item))

## 4.5 Интеграция в гибрид и сравнение с классическими методами

Нейросеть добавлена в гибрид как ещё одна компонента (стандартизованный скор, подбор весов на val, как у остальных), а её top-30 включён в пул кандидатов. Результаты на test - в таблицах раздела 3.1: сравнение «Two-Tower против классических одиночных моделей» и шаги Hybrid-3 -> Hybrid-4 в поэтапном сравнении. Ниже - сводка именно по нейросети.

In [ ]:
nn_rows = ["Content-based (TF-IDF)", "PureSVD (implicit)", "Two-Tower (neural)", "Item-based CF (implicit)",
           "Hybrid-3: + веса по сегментам", "Hybrid-4: + Two-Tower, веса по сегментам", "Hybrid system (пул + баланс)"]
display(results_table(results, ["P@10", "R@10", "nDCG@10", "ild@10", "coverage@10"]).loc[nn_rows])
rows = []
for a, b in [("Two-Tower (neural)", "Content-based (TF-IDF)"), ("Two-Tower (neural)", "PureSVD (implicit)"),
             ("Two-Tower (neural)", "Item-based CF (implicit)"),
             ("Hybrid-4: + Two-Tower, веса по сегментам", "Hybrid-3: + веса по сегментам"),
             ("Hybrid-4: + Two-Tower, веса по сегментам", "Item-based CF (implicit)")]:
    rows.append({"сравнение": f"{a}  -  {b}", **paired_diff(results[a], results[b], "nDCG@10").to_dict()})
display(pd.DataFrame(rows).set_index("сравнение"))

In [ ]:
# качественный пример: один и тот же пользователь, рекомендации разных моделей
u_demo = int(np.flatnonzero(sp.R_trainval.getnnz(axis=1) >= 30)[3]) if (sp.R_trainval.getnnz(axis=1) >= 30).sum() > 3 else int(sp.R_trainval.getnnz(axis=1).argmax())
uid = int(ds.user_ids[u_demo])
liked = sp.trainval_df[(sp.trainval_df["u"] == u_demo) & (sp.trainval_df["rating"] == 5)].head(8)
display(Markdown(f"**Пользователь {uid}**: любимые книги (оценка 5), первые 8"))
display(ds.books.loc[liked["i"], ["title", "authors"]].reset_index(drop=True))
for name, model in [("Item-based CF (implicit)", pipe.models["cf"]), ("Two-Tower", pipe.models.get("neural")),
                    ("Итоговая система", pipe.system)]:
    if model is None:
        continue
    rec = model.rank(np.array([u_demo]), 5, sp.R_trainval[[u_demo]])[0] if hasattr(model, "rank") else model.recommend(np.array([u_demo]), 5, sp.R_trainval)[0]
    display(Markdown(f"*{name}*"))
    display(ds.books.loc[rec, ["title", "authors"]].reset_index(drop=True))

# 5. Сквозной пайплайн

Весь процесс оформлен как класс `Pipeline` (`recsys/pipeline.py`) с параметрами в `PipelineConfig`. Этапы выполняются строго последовательно, и именно их мы вызывали выше по одному:

```
load()            загрузка и проверка схемы CSV, очистка            ─┐
prepare()         разбиение 70/10/20, признаки (книги, пользователи) │ предобработка
fit_train_models()  обучение моделей на train                        │ обучение
tune_hybrid()     подбор весов по типам пользователей (val)          │
tune_balance()    выбор λ (MMR) и доли популярного (val)            ─┘
fit_final()       переобучение на train+val, сборка HybridSystem    ─  генерация рекомендаций
evaluate()        Precision / Recall / nDCG, разнообразие, сегменты ─  оценка качества
export()          рекомендации, метрики, веса, конфигурация, время  ─  артефакты
```

`Pipeline.run()` выполняет всё сразу, `pipe.recommend(user_id, n)` отдаёт рекомендации. Те же этапы запускаются из командной строки:

```bash
python run_pipeline.py --data-dir data --out-dir artifacts/run1          # полный запуск
python run_pipeline.py --data-dir new_data --out-dir artifacts/new       # те же шаги на новых данных
python run_pipeline.py --no-neural --tune-models                         # без нейросети, с подбором гиперпараметров
```

**Перезапуск на новых данных.** Достаточно указать другую папку с `ratings.csv`, `books.csv`, `tags.csv`, `book_tags.csv` в формате Goodbooks-10k: схема проверяется при загрузке, нумерация пользователей и книг, разбиение, признаки, обучение, подбор весов и оценка выполняются заново, ничего не зашито под конкретные данные. Весь случайный выбор зафиксирован `seed`. Ниже этапы сохраняют артефакты, а потом весь пайплайн запускается заново на **другом (синтетическом) наборе данных**.

In [ ]:
pipe.export()
out = Path(pipe.cfg.out_dir)
display(pd.DataFrame({"файл": sorted(p.name for p in out.iterdir()),
                      "размер, КБ": [round(p.stat().st_size / 1024, 1) for p in sorted(out.iterdir())]}))
display(pd.read_csv(out / "recommendations.csv").head(10))

tm = pd.Series(pipe.timings)
fig, ax = plt.subplots(figsize=(9, 3.6))
tm.iloc[::-1].plot.barh(ax=ax, color="C0"); ax.set(xlabel="секунд", title="Время этапов пайплайна")
plt.tight_layout(); plt.show()

In [ ]:
# перезапуск всего пайплайна на новых данных: другой набор (синтетический, другой размер и другие id)
import tempfile
tmp = Path(tempfile.mkdtemp())
synthetic_dataset(tmp / "data", n_users=400, n_books=200, seed=5)
cfg2 = PipelineConfig(
    data_dir=str(tmp / "data"), out_dir=str(tmp / "out"), seed=1,
    params={"popularity": {"strategy": "mean_rating", "min_ratings": 3}, "item_cf": {"k": 5, "shrinkage": 0.0},
            "svd": {"n_factors": 5, "n_epochs": 5, "reg_all": 0.1, "random_state": 0}, "pure_svd": {"n_factors": 30}},
    neural=TwoTowerConfig(epochs=4, batch_size=256, hidden=32, emb_dim=16, out_dim=16, hist_len=20),
    n_topics=6, emb_dim=16, tune_users_per_segment=80, search_random=4, search_sweeps=1, n_export_users=100)
t0 = time.time()
pipe2 = Pipeline(cfg2, verbose=False).run()
print(f"Пайплайн на новых данных выполнен за {time.time() - t0:.0f} с: {pipe2.ds.n_users} пользователей, {pipe2.ds.n_items} книг")
display(results_table(pipe2.results, ["P@10", "R@10", "nDCG@10"]).loc[
    ["Random (reference)", "Item-based CF (implicit)", "Two-Tower (neural)", "Hybrid system (пул + баланс)"]])
display(pipe2.recommend(int(pipe2.ds.user_ids[0]), 5))